In [1]:
from datasets import load_dataset

tiny_stories_instruct = load_dataset('roneneldan/TinyStoriesInstruct')
tiny_stories_instruct

TinyStories-Instruct-train.txt:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

TinyStories-Instruct-valid.txt:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21755681 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/218380 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 21755681
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 218380
    })
})

In [ ]:
END_OF_TEXT = '<|endoftext|>'

def get_story(dataset, index):
    story = []
    start_ind = index
    while True:
        if index >= len(dataset):
            break
        story_line = dataset[index]['text']
        if story_line.strip() == END_OF_TEXT:
            break
        if story_line.strip():
            if index == start_ind and story_line.startswith('Story:'):
                # Remove 'Story:' prefix (if present) in the first line of the Story
                story.append(story_line.removeprefix('Story:'))
            else:
                story.append(story_line)
        index += 1

    return ' '.join(story)

def transform_dataset(dataset):
    # Runtime of approx 4-5 mins in my Mac (CPU) with 32 Gigs of RAM.
    log_intervals = 1000000
    count = 0
    total = len(dataset)
    story_summary_dataset = []
    current_story = ''
    current_summary = ''
    faulty_rows = 0

    for ind, line in enumerate(dataset):
        if line['text'].startswith('Summary:'):
            current_summary = line['text'].removeprefix('Summary: ')
        if line['text'].startswith('Story:'):
            current_story = get_story(dataset, ind)
        if line['text'].strip() == END_OF_TEXT:
            if current_story == '' or current_summary == '':
                print(f'Either Story or Summary is empty: {faulty_rows}')
                faulty_rows+=1
            else:
                story_summary_dataset.append((current_story, current_summary))
            # reset the story and summary since the current item is completed
            current_story = ''
            current_summary = ''

        count += 1
        if count % log_intervals == 0:
            print(f'Count: {count} / {total}, Percentage: {(count/total) * 100.0}')

    return story_summary_dataset

tiny_story_summary_pairs = transform_dataset(tiny_stories_instruct['train'])

In [ ]:
len(tiny_story_summary_pairs)

2476530

In [ ]:
from datasets import Dataset

def prepare_tokenized_dataset(story_summary_pairs):
    items = 500000
    story_summary_pairs_trimmed = story_summary_pairs[:items]
    tokenized_dataset_dict = {
        'story': [story_summary_pair[0].strip() for story_summary_pair in story_summary_pairs_trimmed],
        'summary': [story_summary_pair[1].strip() for story_summary_pair in story_summary_pairs_trimmed]
    }

    hf_dataset = Dataset.from_dict(tokenized_dataset_dict)

    return hf_dataset

In [ ]:
hf_story_summary_dataset = prepare_tokenized_dataset(tiny_story_summary_pairs)

In [ ]:
hf_story_summary_dataset

Dataset({
    features: ['story', 'summary'],
    num_rows: 500000
})

In [ ]:
pwd

In [ ]:
hf_story_summary_dataset.save_to_disk('./datasets/tiny-story-summaries-500k/')

Saving the dataset (2/2 shards): 100%|██████████| 500000/500000 [00:00<00:00, 1926426.32 examples/s]
